<a href="https://colab.research.google.com/github/swapnilsurdi/practice_agentic/blob/main/Week_1_EA_CRM_Lead_Qualifier_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CRM Lead Qualifier Agent

## Goal

To develop an AI agent that automatically enriches a new sales lead (identified by an email address) by gathering publicly available company information, checking for prior engagement in the internal CRM, and assigning a preliminary qualification score.

## Context

Sales representatives often spend valuable time manually researching leads and cross-referencing internal systems before a discovery call. This process is slow, inconsistent, and often leads to a poorly prepared first interaction.

## Agent Functionality

The agent must be able to:

1. Extract Domain: Take the email address and extract the company domain name (e.g., jane@acmecorp.com → acmecorp.com).

2. Enrich Company Data: Use the domain to look up (simulated) company details like industry, size, and annual revenue.

3. Check CRM History: Search the internal (simulated) CRM for any past contact or notes associated with the lead's email.

4. Calculate Lead Score: Synthesize all gathered data to assign a qualitative priority score (e.g., High, Medium, Low).

5. Final Summary: Present a concise, actionable summary of all findings to the sales representative.

# Technical Implementation

We will use the OpenAI client's Function Calling capability to define and execute the necessary business logic tools in a structured loop.

## What is function calling?

Function calling is the mechanism that bridges the gap between an AI model (which is just a text generator) and your actual code (which can perform actions).

Think of the Large Language Model (LLM) as a very smart receptionist. It understands what people want, but it doesn't have the keys to the file cabinet or the ability to make phone calls itself.

Function calling gives the receptionist a "menu" of services it can request from the back office (your code).

![](https://cdn.openai.com/API/docs/images/function-calling-diagram-steps.png)

### The Core Concept

* You provide the tools: You tell the model, "I have a function called get_weather(city) that takes a city name as an argument."

* The model "thinks": If a user asks, "What's the weather in Tokyo?", the model recognizes that your tool can solve this.

* The model outputs JSON (not text): Instead of replying to the user, the model pauses and gives you a structured request: {"function": "get_weather", "arguments": {"city": "Tokyo"}}.

* You execute: Your code sees this request, runs the actual Python function, and gets the result (e.g., "Sunny, 25°C").

* The model finishes: You feed that result back to the model, and it writes the final natural language answer: "It is currently sunny and 25 degrees in Tokyo."

### Why is this powerful?

Without function calling, LLMs are isolated text predictors—they can't "do" anything. With function calling, they become agents.

* **Structured Data Extraction**: Instead of hoping the model formats data correctly, you force it to output clean JSON that fits your database schema.

* **Connecting to the World**: It allows the AI to browse the web, query databases, send emails, or control software, merely by defining those actions as "functions."

In [ ]:
import os
import json
from openai import OpenAI
from google.colab import userdata

In [ ]:
# 1. Initialize OpenAI Client
try:
    client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
except Exception as e:
    print(f"Error initializing OpenAI client: {e}")
    print("Please ensure your OPENAI_API_KEY is set in your environment variables.")

# Defining the "Tools" (Business Logic)

Here we define the actual Python functions that perform the work. In AI Agent terminology, these are the **Tools**.

For this demonstration, we are mocking the external systems (simulating database lookups with dictionaries), but in a production environment, these functions would:
1.  **`lookup_domain_info`**: Query an API like Clearbit or Crunchbase.
2.  **`check_crm_history`**: Query a Salesforce or HubSpot SQL database.
3.  **`calculate_lead_score`**: Run a custom algorithm or ML model to grade the lead.

The agent "calls" these tools by asking us to run these specific Python functions.

In [ ]:
async def lookup_domain_info(domain: str) -> str:
    """
    Looks up and returns mock company information based on its domain.
    In a real system, this would call an external data enrichment API (e.g., Clearbit).
    """
    print(f"-> TOOL ACTIVATED: Looking up domain info for {domain}...")
    await asyncio.sleep(0.5) # Simulate network latency
    # Mock database for demonstration
    mock_data = {
        "acmecorp.com": {"industry": "Software/SaaS", "size": "501-1000 employees", "revenue": "$50M - $100M"},
        "widgetco.net": {"industry": "Manufacturing", "size": "100-250 employees", "revenue": "$10M - $25M"},
        "globalfin.org": {"industry": "Financial Services", "size": "5000+ employees", "revenue": "$1B+"},
    }

    info = mock_data.get(domain, {"industry": "Unknown", "size": "N/A", "revenue": "N/A"})

    # Return the data as a JSON string for the AI model to parse easily
    return json.dumps(info)

In [ ]:
async def check_crm_history(email: str) -> str:
    """
    Checks the internal CRM system for past engagement history with the lead.
    In a real system, this would query a PostgreSQL database or a CRM API (e.g., Salesforce).
    """
    print(f"-> TOOL ACTIVATED: Checking CRM history for {email}...")
    await asyncio.sleep(0.5) # Simulate network latency
    # Mock database for demonstration
    mock_data = {
        "jane@acmecorp.com": {"last_contact": "2025-11-15", "status": "Cold Lead", "notes": "Attended webinar, no follow-up yet."},
        "bob@widgetco.net": {"last_contact": "2025-12-01", "status": "Active Opportunity", "notes": "Discussed Q1 budget and product integration."},
        "default": {"last_contact": "N/A", "status": "No Record", "notes": "New lead, first contact opportunity."},
    }

    history = mock_data.get(email, mock_data["default"])
    return json.dumps(history)


In [ ]:
async def calculate_lead_score(domain_info: str, crm_history: str) -> str:
    """
    Calculates lead score from explicit inputs rather than a bundled JSON string.
    domain_info: JSON string with keys: industry, size, revenue
    crm_history: JSON string with keys: last_contact, status, notes
    """
    print("-> TOOL ACTIVATED: Calculating lead score...")
    await asyncio.sleep(0.5)

    domain = json.loads(domain_info) if isinstance(domain_info, str) else domain_info
    crm = json.loads(crm_history) if isinstance(crm_history, str) else crm_history

    score = "Low"
    if domain.get("revenue", "").startswith("$1B+"):
        score = "High"
    elif crm.get("status") == "Active Opportunity":
        score = "High"
    elif domain.get("revenue", "").startswith("$50M"):
        score = "Medium"

    return json.dumps({"lead_score": score})

In [ ]:
async def identify_domain(email: str) -> str:
    """
    Extracts the domain from the email address.
    """
    print(f"-> TOOL ACTIVATED: Extracting domain from {email}...")

    return json.dumps({"domain": email.split('@')[1]})

In [ ]:
# Map of available function names to the actual Python functions
AVAILABLE_FUNCTIONS = {
    "lookup_domain_info": lookup_domain_info,
    "check_crm_history": check_crm_history,
    "calculate_lead_score": calculate_lead_score,
    "identify_domain": identify_domain
}

# Configuring the Agent's "Menu" (Tool Schema)

The Large Language Model (LLM) cannot see our Python code directly. We must describe our tools to it using a specific JSON format known as a **Schema**.

This schema tells the model:
* **What** the tool does (Description).
* **When** to use it (Context).
* **How** to use it (Parameters/Arguments).

We pass this list to the `tools` parameter in the API call later. It effectively gives the AI a "menu" of actions it can take.

In [ ]:
# This is how the AI model learns about the tools it can use.
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "lookup_domain_info",
            "description": "Retrieves general business information (industry, size, revenue) about a company based on its domain name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "domain": {"type": "string", "description": "The company's domain name, e.g., 'acmecorp.com'"},
                },
                "required": ["domain"],
            },
        },
        "sequence": 0
    },
    {
        "type": "function",
        "function": {
            "name": "check_crm_history",
            "description": "Checks the internal CRM system for past contact, status, and notes associated with a specific lead email.",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {"type": "string", "description": "The full email address of the lead."},
                },
                "required": ["email"],
            },
        },
        "sequence": 0
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_lead_score",
            "description": "Calculates the priority score (High/Medium/Low) for a lead based on a summary of all collected domain and CRM history data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "data_summary": {"type": "string", "description": "A JSON string containing the combined domain_info and crm_history."},
                },
                "required": ["data_summary"],
            },
        },
        "sequence": 0
    },
]

In [ ]:
# from pydantic import BaseModel, Field
# from typing import List, Optional, Dict

# class ToolArguments(BaseModel):
#     # Set all potential arguments as Optional to satisfy the strict schema
#     domain: Optional[str] = Field(None, description="The company domain, e.g., 'acmecorp.com'")
#     email: Optional[str] = Field(None, description="The lead's email address")
#     data_summary: Optional[str] = Field(None, description="Combined JSON data for scoring, include domain_info and crm_history")

# class ToolStep(BaseModel):
#     step_id: str = Field(..., description="Unique ID for this step (e.g., 'step_1')")
#     function_name: str = Field(..., description="One of: 'lookup_domain_info', 'check_crm_history', 'calculate_lead_score', 'identify_domain")
#     arguments: ToolArguments # Use the specific model here instead of Dict
#     depends_on: List[str] = Field(default_factory=list)
#     input_map: Optional[Dict[str, str]] = Field(None, description="Format: {'step_id.key': 'target_param'}")

# class ExecutionPlan(BaseModel):
#     steps: List[ToolStep]

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any

class ToolArguments(BaseModel):
    domain: Optional[str] = Field(
        None,
        description="A company domain name e.g. 'acmecorp.com'. "
                    "Can be wired from identify_domain result via input_map: {'step_N.domain': 'domain'}."
    )
    email: Optional[str] = Field(
        None,
        description="A full email address e.g. 'jane@acmecorp.com'. "
                    "Usually set as a literal from the original user input — no input_map needed."
    )
    domain_info: Optional[str] = Field(
        None,
        description="JSON string of company info required by calculate_lead_score. "
                    "MUST be wired from the entire lookup_domain_info result using input_map "
                    "with just the step_id and NO dot or subkey: {'step_N': 'domain_info'}. "
                    "The runtime serializes the whole result dict automatically. "
                    "Do NOT use 'step_N.industry' or any subkey — pass the whole result."
    )
    crm_history: Optional[str] = Field(
        None,
        description="JSON string of CRM history required by calculate_lead_score. "
                    "MUST be wired from the entire check_crm_history result using input_map "
                    "with just the step_id and NO dot or subkey: {'step_N': 'crm_history'}. "
                    "The runtime serializes the whole result dict automatically. "
                    "Do NOT use 'step_N.status' or any subkey — pass the whole result."
    )

class ToolStep(BaseModel):
    """
    A single node in the execution DAG.

    Dependency & data-passing rules
    --------------------------------
    - `depends_on`: list of step_ids that MUST complete before this step runs.
      Steps with no dependencies run immediately (and in parallel with each other).
    - `input_map`: optional field-level wiring. Format:
        { "<step_id>.<result_key>": "<ToolArguments field name>" }
      Example: { "step_1.domain": "domain" } copies the "domain" key from step_1's
      result into this step's `domain` argument, overriding whatever literal value
      was set in `arguments`.
      Use this whenever a field value is not known at plan-time and must come from
      a prior step's output.
    - If a required argument can be inferred from user input directly (e.g. the raw
      email string), set it as a literal in `arguments` — no input_map needed.
    """
    step_id: str = Field(..., description="Unique identifier, e.g. 'step_1', 'step_2a'.")
    function_name: str = Field(
        ...,
        description="Exact name of the function to call. Must be one of the keys in AVAILABLE_FUNCTIONS."
    )
    arguments: ToolArguments = Field(
        ...,
        description="Literal argument values known at plan-time. Fields that depend on "
                    "prior-step outputs should be left None and wired via input_map instead."
    )
    depends_on: List[str] = Field(
        default_factory=list,
        description="step_ids this step waits for. Empty list = can start immediately."
    )
    input_map: Optional[Dict[str, str]] = Field(
        None,
        description=(
            "Wires prior step outputs into this step's arguments at runtime. "
            "Two valid key formats:\n"
            "  1. '<step_id>.<key>' — extracts a single key from that step's result. "
            "     Example: {'step_1.domain': 'domain'} pulls just the domain string.\n"
            "  2. '<step_id>' — passes the entire result dict, serialized to a JSON string. "
            "     Example: {'step_2': 'domain_info'} passes all of step_2's output as a JSON string.\n"
            "Use format 2 for domain_info and crm_history arguments. "
            "Use format 1 for scalar values like domain or email."
        )
    )

class ExecutionPlan(BaseModel):
    """
    A directed acyclic graph (DAG) of tool calls that the agent will execute.
    The runtime resolves dependencies automatically; steps whose depends_on are
    all satisfied run concurrently.
    """
    steps: List[ToolStep]
    reasoning: str = Field(
        ...,
        description="Brief explanation of why these steps were chosen and in this order. "
                    "If this is a revised plan after errors, explain what changed and why."
    )

In [ ]:
def build_system_prompt(available_functions: dict) -> str:
    fn_list = "\n".join(f"  - {name}" for name in available_functions)
    return (
        "You are an autonomous task-planning agent. "
        "Given a user request, you produce an ExecutionPlan — a DAG of tool calls "
        "that fulfills the request. You never answer directly; you always plan and execute.\n\n"
        f"Available functions:\n{fn_list}\n\n"
        "Consult each function's schema description to understand its inputs and when to use it. "
        "Use depends_on and input_map to wire data between steps. "
        "If you receive error feedback, revise only the failing steps."
    )

In [ ]:
import inspect

async def execute_step(
    step: ToolStep,
    context_results: dict,
    all_context_results: dict,
    completed_steps: set,
    failed_steps: dict,
) -> tuple[bool, Any, Optional[str]]:

    # Wait until each dep is satisfied or failed
    while True:
        satisfied = all(
            dep in completed_steps or
            dep in all_context_results or
            dep in failed_steps
            for dep in step.depends_on
        )
        if satisfied:
            break
        await asyncio.sleep(0.05)

    # Abort if any dependency failed
    broken_deps = [d for d in step.depends_on if d in failed_steps]
    if broken_deps:
        return False, None, f"Skipped: dependencies {broken_deps} failed."

    merged_context = {**all_context_results, **context_results}

    # Start with literal args from the plan
    args = {k: v for k, v in step.arguments.model_dump().items() if v is not None}

    # Resolve input_map
    if step.input_map:
        for source_path, target_field in step.input_map.items():
            try:
                parts = source_path.split(".", 1)
                if len(parts) == 1:
                    # No dot — grab the entire result of that step
                    value = merged_context[parts[0]]
                else:
                    source_step_id, result_key = parts
                    value = merged_context[source_step_id]
                    for part in result_key.split("."):
                        value = value[part]

                # Serialize dicts to JSON string so functions expecting string args work
                if isinstance(value, dict):
                    value = json.dumps(value)

                args[target_field] = value

            except (KeyError, TypeError) as e:
                available_summary = {
                    k: list(v.keys()) if isinstance(v, dict) else type(v).__name__
                    for k, v in merged_context.items()
                }
                err = (
                    f"input_map resolution failed for '{source_path}' -> '{target_field}': {e}. "
                    f"Available steps and their result keys: {json.dumps(available_summary)}"
                )
                return False, None, err

    # Strip args the function doesn't accept, and inject any missing ones it does need
    func = AVAILABLE_FUNCTIONS[step.function_name]
    func_params = inspect.signature(func).parameters
    args = {k: v for k, v in args.items() if k in func_params}

    try:
        raw = await func(**args)
        result = json.loads(raw) if isinstance(raw, str) else raw
        return True, result, None
    except Exception as e:
        full = f"{type(e).__name__}: {e}"
        snippet = full[:300] + (" ... " + full[-100:] if len(full) > 400 else "")
        return False, None, snippet

# The Agent Loop (Think → Act → Observe)

This is the brain of the application. The `run_agent` function implements the core feedback loop required for an autonomous agent.

**How it works:**
1.  **State Management (`collected_data`)**: We initialize a dictionary *outside* the loop to act as the agent's "memory" across multiple steps.
2.  **Think**: We send the conversation history to the model (`client.chat.completions.create`).
3.  **Act**: If the model decides to call a tool, we execute the corresponding Python function.
4.  **Observe**: We append the tool's output to the conversation history as a new message.
5.  **Repeat**: The loop continues until the model decides it has enough information to answer the user directly.



In [ ]:
import asyncio
import json
async def run_agent(user_prompt: str, max_iterations: int = 5):
    print(f"\n--- Agent Starting ---")

    messages = [
        {"role": "system", "content": build_system_prompt(AVAILABLE_FUNCTIONS)},
        {"role": "user", "content": user_prompt},
    ]

    all_context_results = {}   # Accumulates successful step results across iterations

    for iteration in range(max_iterations):
        print(f"\n[Iteration {iteration + 1}] Requesting plan...")

        response = client.beta.chat.completions.parse(
            model="gpt-4o",
            messages=messages,
            response_format=ExecutionPlan,
        )

        plan: Optional[ExecutionPlan] = response.choices[0].message.parsed

        # LLM chose not to produce a plan — it's done
        if plan is None or not plan.steps:
            print("[Agent] No plan produced. Requesting final answer...")
            break

        print(f"[Plan] Reasoning: {plan.reasoning}")
        print(f"[Plan] Steps: {[s.step_id + ':' + s.function_name for s in plan.steps]}")

        # Execute the plan
        context_results = {}   # Only for this iteration
        completed_steps: set = set()
        failed_steps: dict = {}

        async def run_step(step: ToolStep):
            success, result, error = await execute_step(
                step, context_results, all_context_results,  # <-- add this
                completed_steps, failed_steps
            )
            if success:
                context_results[step.step_id] = result
                completed_steps.add(step.step_id)
                print(f"  ✓ {step.step_id} ({step.function_name}): {result}")
            else:
                failed_steps[step.step_id] = error
                print(f"  ✗ {step.step_id} ({step.function_name}): {error}")

        await asyncio.gather(*(run_step(step) for step in plan.steps))

        if failed_steps:
            # Build a compact error summary — don't append the full plan to messages
            error_summary = "; ".join(
                f"{sid}: {err}" for sid, err in failed_steps.items()
            )
            print(f"[Iteration {iteration + 1}] Errors: {error_summary}")

            # Append only a terse user-turn error message — keeps context window clean
            messages.append({
                "role": "user",
                "content": (
                    f"The plan failed on iteration {iteration + 1}. "
                    f"Successful steps: {list(completed_steps) or 'none'}. "
                    f"Errors: {error_summary}. "
                    "Please revise the plan. Only re-plan failing steps; "
                    "successful results are preserved and can be referenced."
                )
            })
            # Carry forward successful results for the next iteration
            all_context_results.update(context_results)

        else:
            # Full success — add the assistant plan + results as a clean turn
            all_context_results.update(context_results)
            messages.append({
                "role": "assistant",
                "content": f"Plan complete. Results: {json.dumps(all_context_results)}"
            })
            messages.append({
                "role": "user",
                "content": "All steps succeeded. Provide the final summary."
            })
            break

    # Final synthesis — now without response_format, so the LLM writes prose
    final = client.chat.completions.create(model="gpt-4o", messages=messages)
    print("\n--- FINAL SUMMARY ---")
    print(final.choices[0].message.content)

# Execution and Testing

Finally, we test our agent with two distinct scenarios to see how it dynamically adapts its behavior.

* **Scenario 1 (Jane@AcmeCorp)**: Represents a high-value enterprise lead. The agent should detect the high revenue and assign a "High" score.
* **Scenario 2 (Bob@WidgetCo)**: Represents a smaller company but with an active deal. The agent should detect the "Active Opportunity" status in the CRM and score it accordingly.

In [ ]:
# Scenario 1: High-Value Lead (Large company, needs scoring)
lead_email_1 = "jane@acmecorp.com"
await run_agent(f"Please qualify this lead for my call tomorrow: {lead_email_1}")

print("\n" + "="*80 + "\n")


--- Agent Starting ---

[Iteration 1] Requesting plan...
[Plan] Reasoning: To qualify the lead based on the provided email 'jane@acmecorp.com', we start by identifying the domain associated with the email. Next, we retrieve company information using the domain and simultaneously check past interactions with this lead via CRM history. Finally, with the company information and CRM history, we calculate the lead score which will help in qualifying the lead for the upcoming call.
[Plan] Steps: ['step_1:identify_domain', 'step_2:lookup_domain_info', 'step_3:check_crm_history', 'step_4:calculate_lead_score']
-> TOOL ACTIVATED: Extracting domain from jane@acmecorp.com...
  ✓ step_1 (identify_domain): {'domain': 'acmecorp.com'}
-> TOOL ACTIVATED: Looking up domain info for acmecorp.com...
-> TOOL ACTIVATED: Checking CRM history for jane@acmecorp.com...
  ✓ step_2 (lookup_domain_info): {'industry': 'Software/SaaS', 'size': '501-1000 employees', 'revenue': '$50M - $100M'}
  ✓ step_3 (check_crm_

In [ ]:
# Scenario 2: Medium-Value Lead (Active opportunity, needs scoring)
lead_email_2 = "bob@widgetco.net"
await run_agent(f"Can you run an analysis on this lead: {lead_email_2}")


--- Agent Starting ---

[Iteration 1] Requesting plan...
[Plan] Reasoning: To analyze the lead associated with the email bob@widgetco.net, we need to perform a series of steps:
1. Identify the domain from the email to understand which company is involved.
2. Use the domain to look up information about the company, which is necessary for calculating the lead score.
3. Check the CRM history associated with the email to retrieve past interactions and activities that inform the lead score.
4. Finally, calculate the lead score by leveraging both the domain info and CRM history.

This sequence ensures that all necessary data is gathered and synthesized to provide an accurate analysis of the lead.
[Plan] Steps: ['step_1:identify_domain', 'step_2:lookup_domain_info', 'step_3:check_crm_history', 'step_4:calculate_lead_score']
-> TOOL ACTIVATED: Extracting domain from bob@widgetco.net...
  ✓ step_1 (identify_domain): {'domain': 'widgetco.net'}
-> TOOL ACTIVATED: Looking up domain info for widge

# Optimization [Optional]

Enhance the CRM Lead Qualifier Agent notebook by adding an introduction to agentic patterns (**Chain-of-Thought vs. ReAct**), implementing detailed JSON logging for tool calls, and visualizing the agentic loop and state management.

## Agentic Patterns: Chain-of-Thought vs. ReAct

To understand how our CRM Lead Qualifier Agent operates, it is essential to distinguish between two primary agentic reasoning patterns: **Chain-of-Thought (CoT)** and **ReAct**.

### 1. Chain-of-Thought (CoT)
Chain-of-Thought is a prompting technique where the model is encouraged to produce intermediate reasoning steps before arriving at a final answer.
- **Process:** Linear and internal.
- **Focus:** Improving logical consistency in complex reasoning tasks (e.g., math or symbolic logic).
- **Limitation:** The model relies entirely on its internal parametric knowledge and cannot interact with the outside world during the process.

### 2. ReAct (Reason + Act)
ReAct combines reasoning traces with task-specific actions. This allows the model to interact with external environments (APIs, databases, tools) and observe the results before deciding on the next step.
- **Process:** Iterative Loop (Think → Act → Observe).
- **Focus:** Dynamic problem-solving where external data is required.
- **Benefit:** Reduces hallucinations by grounding the reasoning in real-world facts retrieved via tools.

### Comparison Table

| Feature | Chain-of-Thought (CoT) | ReAct (Reason + Act) |
| :--- | :--- | :--- |
| **Mechanism** | Sequential internal logic | Interleaved reasoning and tool usage |
| **Data Source** | Internal training data | Real-time external tools/APIs |
| **Feedback** | Static (self-generated) | Dynamic (environment observations) |
| **Ideal Use Case** | Logic puzzles, brainstorming | Search, CRM updates, Data enrichment |

### Why this Agent uses ReAct
The CRM Lead Qualifier Agent specifically follows the **ReAct pattern**. Because the agent must 'know' things it wasn't trained on (like a specific lead's CRM history or current company revenue), it cannot rely on CoT alone. It uses the **Action-Loop** to fetch external data (Observe), analyze that data (Reason), and then decide whether it needs more information or is ready to finalize the lead score.

## Implement Detailed Tool Logging
To get a deeper understanding to how tool calls work, let's modify the tool functions and the `run_agent` loop to print the raw `JSON` messages exchanged between the LLM and the code.


In [ ]:
import json

def lookup_domain_info_v2(domain: str) -> str:
    """Enrich company data and log raw JSON output."""
    mock_data = {
        "acmecorp.com": {"industry": "Software/SaaS", "size": "501-1000 employees", "revenue": "$50M - $100M"},
        "widgetco.net": {"industry": "Manufacturing", "size": "100-250 employees", "revenue": "$10M - $25M"},
        "globalfin.org": {"industry": "Financial Services", "size": "5000+ employees", "revenue": "$1B+"},
    }
    info = mock_data.get(domain, {"industry": "Unknown", "size": "N/A", "revenue": "N/A"})
    json_output = json.dumps(info)
    print(f"[TOOL OUTPUT - lookup_domain_info]: {json_output}")
    return json_output

def check_crm_history_v2(email: str) -> str:
    """Check CRM history and log raw JSON output."""
    mock_data = {
        "jane@acmecorp.com": {"last_contact": "2025-11-15", "status": "Cold Lead", "notes": "Attended webinar, no follow-up yet."},
        "bob@widgetco.net": {"last_contact": "2025-12-01", "status": "Active Opportunity", "notes": "Discussed Q1 budget and product integration."},
        "default": {"last_contact": "N/A", "status": "No Record", "notes": "New lead, first contact opportunity."},
    }
    history = mock_data.get(email, mock_data["default"])
    json_output = json.dumps(history)
    print(f"[TOOL OUTPUT - check_crm_history]: {json_output}")
    return json_output

def calculate_lead_score_v2(data_summary: str) -> str:
    """Calculate lead score and log raw JSON output."""
    data = json.loads(data_summary)
    score = "Low"
    if data["domain_info"].get("revenue", "").startswith("$1B+"):
        score = "High"
    elif data["crm_history"].get("status") == "Active Opportunity":
        score = "High"
    elif data["domain_info"].get("revenue", "").startswith("$50M"):
        score = "Medium"

    result = {"lead_score": score}
    json_output = json.dumps(result)
    print(f"[TOOL OUTPUT - calculate_lead_score]: {json_output}")
    return json_output

# Redefine AVAILABLE_FUNCTIONS to point to updated versions
AVAILABLE_FUNCTIONS_V2 = {
    "lookup_domain_info": lookup_domain_info_v2,
    "check_crm_history": check_crm_history_v2,
    "calculate_lead_score": calculate_lead_score_v2,
}

def run_agent_with_detailed_tool_call(user_prompt: str):
    """Modified agent loop with raw JSON logging for tool calls and responses."""
    print(f"\n--- Running Lead Qualifier Agent with Detailed Logging ---")
    system_prompt = "You are an expert CRM Lead Qualifier Agent. Follow steps: 1. Identify domain 2. lookup_domain_info & check_crm_history 3. Combine JSON 4. calculate_lead_score 5. Summarize."
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    collected_data = {}

    while True:
        print("\n[AI Thinking...]")
        response = client.chat.completions.create(model="gpt-4o", messages=messages, tools=tools_schema, tool_choice="auto")
        response_message = response.choices[0].message
        messages.append(response_message)

        if response_message.tool_calls:
            for tool_call in response_message.tool_calls:
                # LOG: Raw Tool Call from LLM
                print(f"[LLM TOOL CALL]: {json.dumps(tool_call.model_dump(), indent=2)}")

                function_name = tool_call.function.name
                function_to_call = AVAILABLE_FUNCTIONS_V2.get(function_name)
                function_args = json.loads(tool_call.function.arguments)

                if function_name == "calculate_lead_score":
                    function_args = {"data_summary": json.dumps(collected_data)}

                function_result = function_to_call(**function_args)

                if function_name == "lookup_domain_info":
                    collected_data["domain_info"] = json.loads(function_result)
                elif function_name == "check_crm_history":
                    collected_data["crm_history"] = json.loads(function_result)

                tool_message = {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": function_result
                }
                # LOG: Raw Tool response being sent back to LLM
                print(f"[SENT TO LLM]: {json.dumps(tool_message, indent=2)}")
                messages.append(tool_message)
        else:
            print("\n--- FINAL AGENT SUMMARY ---")
            print(response_message.content)
            break

In [ ]:
# Execute test scenario 1 to verify logging
run_agent_with_detailed_tool_call('Please qualify this lead: jane@acmecorp.com')


--- Running Lead Qualifier Agent with Detailed Logging ---

[AI Thinking...]
[LLM TOOL CALL]: {
  "id": "call_LENxIWgnOiHvc1EPm3x1As8z",
  "function": {
    "arguments": "{\"domain\": \"acmecorp.com\"}",
    "name": "lookup_domain_info"
  },
  "type": "function"
}
[TOOL OUTPUT - lookup_domain_info]: {"industry": "Software/SaaS", "size": "501-1000 employees", "revenue": "$50M - $100M"}
[SENT TO LLM]: {
  "role": "tool",
  "tool_call_id": "call_LENxIWgnOiHvc1EPm3x1As8z",
  "name": "lookup_domain_info",
  "content": "{\"industry\": \"Software/SaaS\", \"size\": \"501-1000 employees\", \"revenue\": \"$50M - $100M\"}"
}
[LLM TOOL CALL]: {
  "id": "call_0yCJK5jqbYkLXMfpbAwrUF9T",
  "function": {
    "arguments": "{\"email\": \"jane@acmecorp.com\"}",
    "name": "check_crm_history"
  },
  "type": "function"
}
[TOOL OUTPUT - check_crm_history]: {"last_contact": "2025-11-15", "status": "Cold Lead", "notes": "Attended webinar, no follow-up yet."}
[SENT TO LLM]: {
  "role": "tool",
  "tool_call_i

## Visualizing the ReAct Loop and State Management

To better understand how our agent autonomously navigates through tasks, we can visualize the **ReAct (Reason + Act)** cycle. Unlike a standard script, the agent iteratively determines which tool to use, executes it, and updates its internal 'memory' based on the result.

### The CRM Agent Flowchart

```mermaid
graph TD
    Start([User Input: Lead Email]) --> SystemPrompt[Initialize System Prompt & Empty collected_data]
    
    subgraph ReActLoop [The Agentic Loop]
        Think{LLM: Think} -- Reason: Need Domain Info --> Act1[Act: call lookup_domain_info]
        Think -- Reason: Need CRM History --> Act2[Act: call check_crm_history]
        Think -- Reason: All Data Ready --> Act3[Act: call calculate_lead_score]
        
        Act1 --> Obs1[Observe: Tool Output]
        Act2 --> Obs2[Observe: Tool Output]
        Act3 --> Obs3[Observe: Tool Output]
        
        Obs1 --> UpdateState1[Update collected_data: domain_info]
        Obs2 --> UpdateState2[Update collected_data: crm_history]
        
        UpdateState1 --> Think
        UpdateState2 --> Think
        Obs3 --> FinalThink{Final Synthesis}
    end
    
    subgraph StateManagement [Persistent Memory]
        collected_data[(collected_data Dictionary)]
        UpdateState1 -.->|Store| collected_data
        UpdateState2 -.->|Store| collected_data
        collected_data -.->|Inject into| Act3
    end
    
    FinalThink --> End([Final Agent Summary])
```

### How it Works:
1.  **Think**: The LLM evaluates the current conversation history and decides whether it has enough information to answer or if it needs to use a tool.
2.  **Act**: The agent triggers a specific Python function (e.g., `lookup_domain_info`). Note that for `calculate_lead_score`, the agent passes the entire accumulated `collected_data` object.
3.  **Observe**: The output of the function is returned to the LLM as a `tool` role message.
4.  **State Management**: Throughout this cycle, the `collected_data` dictionary acts as the agent's **short-term memory**, ensuring that data gathered in step 1 (Domain) is still available for step 4 (Scoring) even if the LLM 'forgets' specific values in a long conversation context.

## Summary:

### Q&A

**What are the primary technical enhancements made to the CRM Lead Qualifier Agent?**
The notebook was upgraded with a consolidated agent environment that implements detailed JSON logging for both LLM tool calls and tool outputs. Additionally, a ReAct (Reason + Act) visualization was integrated to explain the agentic loop and state management.

**How does the agent handle state across multiple tool calls?**
The agent uses a persistent `collected_data` dictionary. As the agent iterates through the ReAct loop, it stores outputs from domain lookups and CRM history checks into this dictionary, which is then injected into the final lead scoring tool to ensure data integrity.

**What are the primary agentic patterns discussed and implemented?**
The task focuses on comparing and implementing Chain-of-Thought (CoT) and ReAct (Reasoning and Acting) patterns. CoT focuses on the internal reasoning steps of the model, while ReAct integrates that reasoning with external tool usage in a continuous loop.

**How is the ReAct loop monitored and visualized?**
The process involves implementing detailed JSON logging for all tool calls and responses to provide transparency. Additionally, a Mermaid.js diagram is used to visualize the state management and the iterative nature of the ReAct cycle.

*   **Logical Dependency Failure:** The workflow relies on a sequential dependency where `domain_info` must be fetched first to serve as an input for lead scoring.
*   **Missing State Validation:** The execution logic did not verify the existence of required keys in the `collected_data` dictionary before proceeding to the next analytical step.
*   **Race Condition/Order of Operations:** The `KeyError` highlights that the agent bypassed a necessary data acquisition step (`lookup_domain_info`), resulting in a runtime crash when the subsequent tool tried to access non-existent data.  

*  **What can cause a `KeyError: 'domain_info'` during the execution of a cell:** The error can occur when the agent attempts to execute the `calculate_lead_score` tool before the `lookup_domain_info` tool had finished its execution. As a result, the required `domain_info` key may not yet have been populated in the `collected_data` shared memory dictionary, leading to a lookup failure. Pydantic validation may be effective here.

### Data Analysis Key Findings

*   **Standardized Tool Integration**: Three core tools (`lookup_domain_info`, `check_crm_history`, and `calculate_lead_score`) were successfully mapped to a `tools_schema`, allowing the GPT-4o model to execute them sequentially. By creating a consolidated agent environment, the system ensures that diverse tools (e.g., lead scoring, database queries) return standardized JSON outputs, which reduces parsing errors by the LLM.
*   **Enhanced Observability/Detailed Execution Logging**: The updated `run_agent` loop provides full transparency by printing raw JSON messages. This includes the `LLM TOOL CALL` (function name and arguments) and the `SENT TO LLM` response (tool output and unique `tool_call_id`). The shift to structured JSON logging allows for programmatic auditing of the agent's "thought process," making it easier to identify where a lead qualification might have failed.
*   **Stateful Memory Management/Pattern**: By using a centralized `collected_data` object, the agent successfully scored a test lead (`jane@acmecorp.com`) as "Medium" based on business data ($50M+ revenue) gathered in a previous step of the loop. Implementing stateful memory ensures the agent retains context across multiple tool calls, preventing redundant operations and reducing API token costs associated with re-sending the entire history.
*   **ReAct Architecture Visualization**: The implementation follows a formal Reason-Act-Observe pattern. The visualization confirms that the agent does not just follow a linear script but evaluates the context at each turn to decide the next best action.
*   **Agentic Pattern Distinction:** The implementation highlights that while Chain-of-Thought improves logical consistency in complex tasks, the ReAct pattern is essential for tasks requiring real-time data or interaction with external systems (like a CRM).


### Insights or Next Steps

*   **Error Handling/Resilience and Validation**: Future iterations should include robust error handling for API failures and data validation schemas (e.g., Pydantic) to ensure the `collected_data` maintains a consistent structure. It should focus on "Self-Correction" patterns, where the agent is prompted to analyze its own JSON error logs to retry tool calls with corrected parameters.
*   **Asynchronous Execution**: To improve performance, the agent loop could be updated to support parallel tool execution for independent tasks like domain lookups and CRM history checks.
*   **Evaluation Framework:** The next logical step is to implement a benchmark or evaluation set to quantify the accuracy improvement of the ReAct pattern versus a standard zero-shot prompt in lead qualification.

*   **Implement Pre-checks:** Future tool calls should include a validation step to ensure all required keys exist in the shared memory state before execution.
*   **Sequential Logic Enforcement:** Ensure the agent's planning module explicitly recognizes that the output of the domain lookup is a mandatory prerequisite for the lead scoring calculation.





